In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
(full_train_images, full_train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
full_train_images = full_train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0
full_train_labels = full_train_labels.flatten()
test_labels = test_labels.flatten()

train_images, val_images = full_train_images[:45000], full_train_images[45000:]
class_names = ["airplane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
print(train_images.shape, val_images.shape, test_images.shape)

fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
for axis, index in zip(axes.flatten(), range(20)):
    axis.imshow(train_images[index])
    axis.set_title(class_names[full_train_labels[index]], fontsize=8)
    axis.axis("off")
plt.show()

In [ ]:
def build_dense_autoencoder(latent_dim=256):
    encoder_input = keras.Input(shape=(32, 32, 3))
    x = layers.Flatten()(encoder_input)
    x = layers.Dense(1024, activation="relu")(x)
    x = layers.Dense(512, activation="relu")(x)
    latent_code = layers.Dense(latent_dim, activation="relu", name="latent")(x)
    encoder = keras.Model(encoder_input, latent_code, name="dense_encoder")

    decoder_input = keras.Input(shape=(latent_dim,))
    y = layers.Dense(512, activation="relu")(decoder_input)
    y = layers.Dense(1024, activation="relu")(y)
    y = layers.Dense(32 * 32 * 3, activation="sigmoid")(y)
    decoder_output = layers.Reshape((32, 32, 3))(y)
    decoder = keras.Model(decoder_input, decoder_output, name="dense_decoder")

    autoencoder = keras.Model(encoder_input, decoder(encoder(encoder_input)), name="dense_autoencoder")
    autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return encoder, decoder, autoencoder

def build_conv_autoencoder(latent_dim=256):
    encoder_input = keras.Input(shape=(32, 32, 3))
    x = encoder_input
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, strides=2, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
    x = layers.Flatten()(x)
    latent_code = layers.Dense(latent_dim, name="latent")(x)
    encoder = keras.Model(encoder_input, latent_code, name="conv_encoder")

    decoder_input = keras.Input(shape=(latent_dim,))
    y = layers.Dense(4 * 4 * 128)(decoder_input)
    y = layers.Reshape((4, 4, 128))(y)
    for filters in [128, 64, 32]:
        y = layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(y)
        y = layers.BatchNormalization()(y)
        y = layers.LeakyReLU(0.2)(y)
    decoder_output = layers.Conv2D(3, 3, padding="same", activation="sigmoid")(y)
    decoder = keras.Model(decoder_input, decoder_output, name="conv_decoder")

    autoencoder = keras.Model(encoder_input, decoder(encoder(encoder_input)), name=f"conv_autoencoder_{latent_dim}")
    autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return encoder, decoder, autoencoder

dense_encoder, dense_decoder, dense_autoencoder = build_dense_autoencoder()
conv_encoder, conv_decoder, conv_autoencoder = build_conv_autoencoder()
conv_encoder.summary()
conv_decoder.summary()
print("dense autoencoder parameters:", dense_autoencoder.count_params())
print("conv autoencoder parameters:", conv_autoencoder.count_params())

In [ ]:
reconstruction_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
]

dense_history = dense_autoencoder.fit(train_images, train_images, validation_data=(val_images, val_images), epochs=25, batch_size=128, callbacks=reconstruction_callbacks)

In [ ]:
conv_history = conv_autoencoder.fit(train_images, train_images, validation_data=(val_images, val_images), epochs=25, batch_size=128, callbacks=reconstruction_callbacks)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(dense_history.history["loss"], label="dense train")
plt.plot(dense_history.history["val_loss"], label="dense validation")
plt.plot(conv_history.history["loss"], label="conv train")
plt.plot(conv_history.history["val_loss"], label="conv validation")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("reconstruction MSE")
plt.title("Reconstruction loss")
plt.legend()
plt.show()

In [ ]:
sample_count = 2000
test_sample = test_images[:sample_count]
dense_reconstructions = dense_autoencoder.predict(test_sample, verbose=0)
conv_reconstructions = conv_autoencoder.predict(test_sample, verbose=0)

def reconstruction_metrics(originals, reconstructions):
    return {
        "mse": float(np.mean((originals - reconstructions) ** 2)),
        "psnr": float(tf.reduce_mean(tf.image.psnr(originals, reconstructions, max_val=1.0))),
        "ssim": float(tf.reduce_mean(tf.image.ssim(originals, reconstructions, max_val=1.0)))
    }

metric_frame = pd.DataFrame({
    "dense autoencoder": reconstruction_metrics(test_sample, dense_reconstructions),
    "conv autoencoder": reconstruction_metrics(test_sample, conv_reconstructions)
}).T
display(metric_frame.round(4))

In [ ]:
shown_images = 10
fig, axes = plt.subplots(3, shown_images, figsize=(18, 6))
for column in range(shown_images):
    for row, (images, title) in enumerate([(test_sample, "original"), (dense_reconstructions, "dense"), (conv_reconstructions, "conv")]):
        axes[row, column].imshow(np.clip(images[column], 0, 1))
        axes[row, column].axis("off")
        if column == 0:
            axes[row, column].set_title(title, loc="left")
plt.tight_layout()
plt.show()

In [ ]:
latent_results = {}
for latent_dim in [16, 64, 128, 256, 512]:
    keras.utils.set_random_seed(42)
    _, _, candidate_autoencoder = build_conv_autoencoder(latent_dim)
    candidate_history = candidate_autoencoder.fit(train_images, train_images, validation_data=(val_images, val_images), epochs=12, batch_size=128, verbose=0)
    candidate_reconstructions = candidate_autoencoder.predict(test_sample, verbose=0)
    latent_results[latent_dim] = {
        "history": candidate_history.history,
        "reconstructions": candidate_reconstructions[:8],
        **reconstruction_metrics(test_sample, candidate_reconstructions)
    }
    print(f"latent {latent_dim:<4} compression {3072 / latent_dim:>6.1f}x  test mse {latent_results[latent_dim]['mse']:.5f}  psnr {latent_results[latent_dim]['psnr']:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for latent_dim, result in latent_results.items():
    axes[0].plot(result["history"]["val_loss"], label=f"latent {latent_dim}")
axes[0].set_yscale("log")
axes[0].set_title("Validation reconstruction loss by latent size")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].bar([str(latent_dim) for latent_dim in latent_results], [result["mse"] for result in latent_results.values()], color="teal")
axes[1].set_title("Test MSE by latent size")
axes[1].set_xlabel("latent dimension")
plt.show()

fig, axes = plt.subplots(len(latent_results) + 1, 8, figsize=(14, 2 * (len(latent_results) + 1)))
for column in range(8):
    axes[0, column].imshow(test_sample[column])
    axes[0, column].axis("off")
    for row, (latent_dim, result) in enumerate(latent_results.items(), start=1):
        axes[row, column].imshow(np.clip(result["reconstructions"][column], 0, 1))
        axes[row, column].axis("off")
axes[0, 0].set_title("original", loc="left")
for row, latent_dim in enumerate(latent_results, start=1):
    axes[row, 0].set_title(f"latent {latent_dim}", loc="left")
plt.tight_layout()
plt.show()

In [ ]:
per_image_error = np.mean((test_sample - conv_reconstructions) ** 2, axis=(1, 2, 3))
sample_labels = test_labels[:sample_count]
class_error = [per_image_error[sample_labels == class_id].mean() for class_id in range(10)]

plt.figure(figsize=(10, 4))
plt.bar(class_names, class_error, color="slateblue")
plt.title("Conv autoencoder reconstruction error per class")
plt.ylabel("MSE")
plt.show()

hardest = np.argsort(-per_image_error)[:6]
easiest = np.argsort(per_image_error)[:6]
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for column in range(6):
    for row, (images, index_list) in enumerate([(test_sample, easiest), (conv_reconstructions, easiest), (test_sample, hardest), (conv_reconstructions, hardest)]):
        axes[row, column].imshow(np.clip(images[index_list[column]], 0, 1))
        axes[row, column].axis("off")
axes[0, 0].set_title("easiest originals", loc="left")
axes[2, 0].set_title("hardest originals", loc="left")
plt.tight_layout()
plt.show()

In [ ]:
first_code, second_code = conv_encoder.predict(test_sample[[3, 7]], verbose=0)
blend_steps = np.linspace(0, 1, 10)
blended_codes = np.array([(1 - step) * first_code + step * second_code for step in blend_steps])
blended_images = conv_decoder.predict(blended_codes, verbose=0)

fig, axes = plt.subplots(1, len(blend_steps), figsize=(18, 2.5))
for axis, image, step in zip(axes, blended_images, blend_steps):
    axis.imshow(np.clip(image, 0, 1))
    axis.set_title(f"{step:.1f}")
    axis.axis("off")
plt.suptitle("Interpolating between two images in latent space")
plt.show()

conv_autoencoder.save("conv_autoencoder.keras")